# Register a model in the MLflow server

We will use the MlflowClient instance to:

- Register a new version for the experiment green-taxi-trip-duration
- Retrieve the latests versions of the model 
- Transition to "production" and adding annotations to it.

The environment variables are set in the a `.env` file. 
Create one and add yours.
+ `MLFLOW_TRACKING_URI` : the tracking server URI
+ `RUN_ID` : the run id of the experiment you want to register as a model
+ `MODEL_NAME` : the name you gave to the model (you can check it in the MLflow UI)

In [ ]:
import mlflow
from mlflow import MlflowClient
import os 
from dotenv import load_dotenv
from datetime import datetime


In [ ]:
load_dotenv()
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")
RUN_ID = os.environ.get("RUN_ID")
MODEL_NAME = os.environ.get("MODEL_NAME")

Connect to the MLFlow server:

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

Register the model:

In [ ]:
model_uri = f"runs:/{RUN_ID}/{MODEL_NAME}"
client = MlflowClient()
# Create registered model
tags = {"LinearRegression.framework": "Scikit-Learn"}
desc = "Model to predict ride duration in NYC"

client.create_registered_model(name='lr-ride-duration',tags=tags,description=desc)

Create a model version:

In [ ]:
model_version = client.create_model_version(
    name="lr-ride-duration",
    source=model_uri,
    run_id=RUN_ID
)

print(f"Model registered: {model_version.name} v{model_version.version}")


You now also see the model in the MLflow UI under `Models`.
![](../images/mlflow_model_registered.png)

In [ ]:
model_name = "lr-ride-duration"
# List all model versions
versions = client.search_model_versions(filter_string=f"name='{model_name}'")
for version in versions:
    print(f"version: {version.version}, source: {version.source}, alias: {version.aliases}")

We see we only have a version 1 of the model. If we would have registered more artifacts as models under the same name, we would have seen more versions here.
But we want to change the alias to `production` and add a description to it. 

First change the alias of the version to `production`:

In [ ]:
model_version = 1
new_alias = "production"
client.set_registered_model_alias(
    name=model_name,
    version=model_version,
    alias=new_alias
)

And now add the description:

In [ ]:
date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_alias} on {date}"
)

![](../images/mlflow_model_registered_description.png)